# RAG Chatbot PDF

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from dataclasses import dataclass
from typing import Tuple

# CONFIGURATION
@dataclass(frozen=True)
class RAGConfig:
    """Immutable configuration for the RAG pipeline."""
    chunk_size: int = 1000
    chunk_overlap: int = 150
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    llm_model: str = "llama-3.1-8b-instant"
    temperature: float = 0.2
    top_k_retrieval: int = 10
    top_k_compression: int = 5
    max_tokens: int = 4096
    allowed_extensions: Tuple[str, ...] = ('.pdf', '.txt')
    max_file_size_mb: int = 50
    log_level: str = "INFO"


In [3]:
import logging

# LOGGING SETUP

def setup_logging(level: str = "INFO") -> logging.Logger:
    """Configure structured logging for the RAG system."""
    logger = logging.getLogger("RAGSystem")
    logger.setLevel(getattr(logging, level.upper()))
    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger


In [4]:
import re
import hashlib
from pathlib import Path

# SECURITY UTILITIES


class SecurityValidator:
    """Input validation and sanitization to prevent common vulnerabilities."""

    @staticmethod
    def sanitize_filename(filename: str) -> str:
        
        if not filename or not isinstance(filename, str):
            raise ValueError("Invalid filename provided")

        # Remove path separators and null bytes
        sanitized = re.sub(r'[\\/<>\":|?*\x00-\x1f]', '_', filename)
        sanitized = sanitized.strip('. ')

        if not sanitized or sanitized in ('.', '..'):
            raise ValueError("Filename became invalid after sanitization")

        # Add hash prefix to prevent collision attacks
        name_hash = hashlib.sha256(filename.encode()).hexdigest()[:8]
        stem = Path(sanitized).stem
        suffix = Path(sanitized).suffix
        return f"{stem}_{name_hash}{suffix}"

    @staticmethod
    def validate_file_size(content: bytes, max_mb: int = 50) -> None:
        size_mb = len(content) / (1024 * 1024)
        if size_mb > max_mb:
            raise ValueError(f"File size ({size_mb:.1f}MB) exceeds limit ({max_mb}MB)")

    @staticmethod
    def validate_extension(filename: str, allowed: Tuple[str, ...]) -> None:
        ext = Path(filename).suffix.lower()
        if ext not in allowed:
            raise ValueError(f"Extension '{ext}' not allowed. Use: {allowed}")

    @staticmethod
    def mask_api_key(key: str) -> str:
        if len(key) <= 8:
            return "***"
        return f"{key[:4]}...{key[-4:]}"




In [5]:
from typing import List
from datetime import datetime
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# DOCUMENT PROCESSING

class DocumentProcessor:

    def __init__(self, config: RAGConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

    def load_document(self, file_path: Path) -> List[Document]:
        ext = file_path.suffix.lower()
        self.logger.info(f"Loading document: {file_path.name}")

        try:
            if ext == '.pdf':
                loader = PyPDFLoader(str(file_path), extract_images=False)
                docs = loader.load()
            elif ext == '.txt':
                loader = TextLoader(str(file_path), encoding="utf-8")
                docs = loader.load()
            else:
                raise ValueError(f"Unsupported file type: {ext}")

            # Enrich metadata with source info
            for i, doc in enumerate(docs):
                doc.metadata.update({
                    "source_file": file_path.name,
                    "page_number": doc.metadata.get("page", i + 1),
                    "chunk_index": i,
                    "processed_at": datetime.now().isoformat()
                })

            self.logger.info(f"Loaded {len(docs)} pages from {file_path.name}")
            return docs

        except Exception as e:
            self.logger.error(f"Failed to load {file_path.name}: {e}")
            raise RuntimeError(f"Document loading failed: {e}")

    def split_documents(self, documents: List[Document]) -> List[Document]:
        self.logger.info(f"Splitting {len(documents)} documents into chunks...")
        chunks = self.text_splitter.split_documents(documents)

        # Add chunk-level metadata
        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = i
            chunk.metadata["total_chunks"] = len(chunks)

        self.logger.info(f"Created {len(chunks)} chunks")
        return chunks



In [6]:
import os
from typing import Optional, Any
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.document_compressors import DocumentCompressorPipeline

# VECTOR STORE & RETRIEVAL

class VectorStoreManager:
    """Manages embeddings, vector store creation, and advanced retrieval."""

    def __init__(self, config: RAGConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.embeddings = HuggingFaceEmbeddings(
            model_name=config.embedding_model,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
        self.vector_store: Optional[FAISS] = None
        self.retriever: Optional[Any] = None

    def build_index(self, documents: List[Document]) -> None:
       
        self.logger.info("Building vector index...")

        try:
            self.vector_store = FAISS.from_documents(documents, self.embeddings)

            # Build contextual compression retriever for better results
            base_retriever = self.vector_store.as_retriever(
                search_type="similarity",
                search_kwargs={"k": self.config.top_k_retrieval}
            )

            # Remove redundant documents and rerank
            redundant_filter = EmbeddingsRedundantFilter(
                embeddings=self.embeddings,
                similarity_threshold=0.95
            )
            pipeline = DocumentCompressorPipeline(transformers=[redundant_filter])

            self.retriever = ContextualCompressionRetriever(
                base_compressor=pipeline,
                base_retriever=base_retriever
            )

            self.logger.info("Vector index built successfully with compression")

        except Exception as e:
            self.logger.error(f"Index build failed: {e}")
            raise RuntimeError(f"Vector store creation failed: {e}")

    def save_index(self, path: str) -> None:
        if self.vector_store:
            self.vector_store.save_local(path)
            self.logger.info(f"Index saved to {path}")

    def load_index(self, path: str) -> None:
        if os.path.exists(path):
            self.vector_store = FAISS.load_local(
                path, self.embeddings, allow_dangerous_deserialization=True
            )
            self.logger.info(f"Index loaded from {path}")



In [7]:
from typing import Optional, Any, Dict
from langchain_groq import ChatGroq
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate


# LLM & CHAIN


class RAGChain:
    """Manages LLM configuration, prompt engineering, and conversational chain."""

    SYSTEM_PROMPT = """You are an expert research assistant. Your task is to provide accurate, 
comprehensive, and well-structured answers based on the provided document context.

CRITICAL RULES:
1. Base your answer ONLY on the provided context. If the context doesn't contain the answer, say so explicitly.
2. Cite your sources using [Source: filename, Page X] format after each claim.
3. Structure your response with clear headings, bullet points, and bold text for readability.
4. Be thorough but concise. Avoid hallucination or adding information not in the context.
5. If the question is ambiguous, ask for clarification.

Context:
{context}

Chat History:
{chat_history}

Question: {question}

Provide a detailed, well-cited answer:"""

    def __init__(self, config: RAGConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.llm: Optional[ChatGroq] = None
        self.chain: Optional[Any] = None
        self.memory: Optional[ConversationBufferMemory] = None

    def initialize(self, retriever: Any) -> None:
        """
        Initialize LLM and conversational chain with memory.

        Args:
            retriever: Configured retriever from VectorStoreManager
        """
        self.logger.info(f"Initializing LLM: {self.config.llm_model}")

        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            raise ValueError("GROQ_API_KEY not found in environment")

        self.llm = ChatGroq(
            model=self.config.llm_model,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
            api_key=api_key
        )

        # Conversation memory with explicit output key
        self.memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key="answer",
            input_key="question"
        )

        prompt = PromptTemplate(
            template=self.SYSTEM_PROMPT,
            input_variables=["context", "chat_history", "question"]
        )

        self.chain = ConversationalRetrievalChain.from_llm(
            llm=self.llm,
            retriever=retriever,
            memory=self.memory,
            combine_docs_chain_kwargs={"prompt": prompt},
            return_source_documents=True,
            verbose=False
        )

        self.logger.info("RAG chain initialized with conversation memory")

    def invoke(self, question: str) -> Dict[str, Any]:
        if not self.chain:
            raise RuntimeError("Chain not initialized. Call initialize() first.")

        self.logger.info(f"Processing query: {question[:50]}...")

        try:
            result = self.chain.invoke({"question": question})
            return {
                "answer": result.get("answer", "No answer generated"),
                "sources": result.get("source_documents", [])
            }
        except Exception as e:
            self.logger.error(f"Query failed: {e}")
            raise RuntimeError(f"Failed to process query: {e}")

    def reset_memory(self) -> None:
        """Clear conversation history."""
        if self.memory:
            self.memory.clear()
            self.logger.info("Conversation memory cleared")

In [8]:
import tempfile
from contextlib import contextmanager
from typing import Optional, Dict, Any

# ORCHESTRATOR
class RAGSystem:
    """
    Main orchestrator that coordinates all components.
    Provides a clean interface for the UI layer.
    """

    def __init__(self, config: Optional[RAGConfig] = None):
        self.config = config or RAGConfig()
        self.logger = setup_logging(self.config.log_level)
        self.security = SecurityValidator()
        self.processor = DocumentProcessor(self.config, self.logger)
        self.vector_manager = VectorStoreManager(self.config, self.logger)
        self.chain = RAGChain(self.config, self.logger)
        self.is_ready: bool = False
        self.current_file: Optional[str] = None

    @contextmanager
    def _temp_file(self, filename: str, content: bytes):
        """Context manager for safe temporary file handling."""
        safe_name = self.security.sanitize_filename(filename)
        temp_path = Path(tempfile.gettempdir()) / safe_name
        try:
            temp_path.write_bytes(content)
            self.logger.info(f"Temp file created: {temp_path}")
            yield temp_path
        finally:
            if temp_path.exists():
                temp_path.unlink()
                self.logger.info(f"Temp file cleaned: {temp_path}")

    def build_pipeline(self, api_key: str, uploaded_file: Dict[str, Any]) -> None:
        """
        Build complete RAG pipeline from uploaded file.

        Args:
            api_key: Groq API key
            uploaded_file: Dictionary with name and content keys
        """
        # Validate inputs
        if not api_key or len(api_key) < 10:
            raise ValueError("Invalid API key provided")

        os.environ["GROQ_API_KEY"] = api_key
        self.logger.info(f"API key set: {self.security.mask_api_key(api_key)}")

        filename = uploaded_file.get("name", "")
        content = uploaded_file.get("content", b"")

        self.security.validate_extension(filename, self.config.allowed_extensions)
        self.security.validate_file_size(content, self.config.max_file_size_mb)

        # Process document safely
        with self._temp_file(filename, content) as temp_path:
            documents = self.processor.load_document(temp_path)
            chunks = self.processor.split_documents(documents)

            self.vector_manager.build_index(chunks)
            self.chain.initialize(self.vector_manager.retriever)

            self.current_file = filename
            self.is_ready = True
            self.logger.info("Pipeline build complete")

    def query(self, question: str) -> Dict[str, Any]:
        """Process a user query through the RAG pipeline."""
        if not self.is_ready:
            raise RuntimeError("Pipeline not built. Upload a document first.")
        if not question or not question.strip():
            raise ValueError("Empty question provided")

        return self.chain.invoke(question.strip())

    def reset(self) -> None:
        """Reset system state."""
        self.chain.reset_memory()
        self.is_ready = False
        self.current_file = None
        self.logger.info("System reset")



In [9]:
from typing import Optional
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML

# UI LAYER
class RAGInterface:

    def __init__(self):
        self.system: Optional[RAGSystem] = None
        self._build_ui()

    def _build_ui(self) -> None:
        """Construct the widget interface."""
        # Setup panel
        self.api_input = widgets.Password(
            description="Groq API Key:",
            layout=widgets.Layout(width='400px'),
            style={'description_width': '120px'}
        )
        self.uploader = widgets.FileUpload(
            accept='.pdf,.txt',
            multiple=False,
            description="📄 Upload Document",
            layout=widgets.Layout(width='300px')
        )
        self.build_btn = widgets.Button(
            description="🚀 Build Pipeline",
            button_style='success',
            layout=widgets.Layout(width='200px')
        )
        self.reset_btn = widgets.Button(
            description="🔄 Reset",
            button_style='warning',
            layout=widgets.Layout(width='120px'),
            disabled=True
        )

        self.status_out = widgets.Output()
        self.chat_out = widgets.Output(
            layout={
                'border': '1px solid #ddd',
                'height': '500px',
                'overflow': 'auto',
                'padding': '20px',
                'background': '#fafafa'
            }
        )
        self.chat_input = widgets.Text(
            placeholder='Ask about your document...',
            layout=widgets.Layout(width='75%'),
            disabled=True
        )
        self.send_btn = widgets.Button(
            description='➤ Ask',
            button_style='primary',
            layout=widgets.Layout(width='100px'),
            disabled=True
        )

        # Bind events
        self.build_btn.on_click(self._on_build)
        self.reset_btn.on_click(self._on_reset)
        self.send_btn.on_click(self._on_send)
        self.chat_input.on_submit(self._on_send)

        # Display initial layout
        setup_box = widgets.HBox([
            widgets.VBox([self.api_input, self.uploader]),
            widgets.VBox([self.build_btn, self.reset_btn])
        ])

        chat_box = widgets.VBox([
            self.chat_out,
            widgets.HBox([self.chat_input, self.send_btn], 
                        layout=widgets.Layout(justify_content='center', margin='10px'))
        ])

        display(widgets.VBox([
            widgets.HTML("<h2>🤖 Production RAG Chatbot</h2>"),
            setup_box,
            self.status_out,
            chat_box
        ]))

    def _log_status(self, message: str, level: str = "info") -> None:
        colors = {"info": "#2196F3", "success": "#4CAF50", "error": "#f44336", "warning": "#FF9800"}
        icon = {"info": "ℹ️", "success": "✅", "error": "❌", "warning": "⚠️"}
        with self.status_out:
            display(HTML(
                f'<div style="padding:8px 12px; margin:5px 0; '
                f'border-left:4px solid {colors.get(level, "#333")}; '
                f'background:#f5f5f5;">'
                f'{icon.get(level, "•")} <b>{message}</b></div>'
            ))

    def _on_build(self, b) -> None:
        """Handle pipeline build button click."""
        with self.status_out:
            clear_output()

            if not self.api_input.value:
                self._log_status("API key is required", "error")
                return
            if not self.uploader.value:
                self._log_status("Please upload a document", "error")
                return

            try:
                uploaded = self.uploader.value[0]
                self.system = RAGSystem()

                self._log_status(f"Processing '{uploaded['name']}'...", "info")
                self.system.build_pipeline(self.api_input.value, uploaded)

                self._log_status(
                    f"Pipeline ready! Document: {uploaded['name']}", 
                    "success"
                )

                # Enable chat controls
                self.chat_input.disabled = False
                self.send_btn.disabled = False
                self.reset_btn.disabled = False
                self.build_btn.disabled = True
                self.uploader.disabled = True
                self.api_input.disabled = True

                # Welcome message
                with self.chat_out:
                    display(Markdown(
                        "🤖 **System:** Ready! I have access to your document. "
                        "Ask me anything, and I'll cite my sources."
                    ))
                    display(Markdown("---"))

            except Exception as e:
                self._log_status(f"Build failed: {str(e)}", "error")
                self.system = None

    def _on_send(self, b=None) -> None:
        """Handle send button click with streaming simulation."""
        question = self.chat_input.value.strip()
        if not question:
            return

        self.chat_input.value = ""
        self.chat_input.disabled = True
        self.send_btn.disabled = True

        try:
            with self.chat_out:
                # User message
                display(Markdown(f"👤 **You:** {question}"))

                # AI response with simulated typing
                response = self.system.query(question)
                answer = response["answer"]
                sources = response.get("sources", [])

                # Format answer with sources
                display(Markdown(f"🤖 **AI:**\n{answer}"))

                # Source citations
                if sources:
                    source_texts = []
                    seen = set()
                    for src in sources[:3]:  # Top 3 sources
                        meta = src.metadata
                        key = (meta.get("source_file"), meta.get("page_number"))
                        if key not in seen and None not in key:
                            seen.add(key)
                            source_texts.append(
                                f"- **{meta['source_file']}** (Page {meta['page_number']})"
                            )

                    if source_texts:
                        display(Markdown(
                            "\n📚 **Sources:**\n" + "\n".join(source_texts)
                        ))

                display(Markdown("---"))


        except Exception as e:
            with self.chat_out:
                display(Markdown(f"❌ **Error:** {str(e)}"))
                display(Markdown("---"))
        finally:
            self.chat_input.disabled = False
            self.send_btn.disabled = False
            self.chat_input.focus()

    def _on_reset(self, b) -> None:
        """Handle reset button click."""
        if self.system:
            self.system.reset()
            self.system = None

        self.chat_out.clear_output()
        with self.status_out:
            clear_output()
            self._log_status("System reset. Upload a new document.", "info")

        self.chat_input.disabled = True
        self.send_btn.disabled = True
        self.reset_btn.disabled = True
        self.build_btn.disabled = False
        self.uploader.disabled = False
        self.api_input.disabled = False
        self.uploader.value = []



# LAUNCH


# Auto-launch when cell runs
interface = RAGInterface()
